# Build: the capstone backend API surface

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 25 §25.5 · type: walkthrough*  🔧 *This is the chapter's Build.*

**The promise:** you will assemble the capstone's small, deliberately **thin** public surface — `POST /runs` (enqueue, return a `run_id`), `GET /runs/{id}` (status + result), `GET /runs/{id}/stream` (SSE progress), `POST /ask` (sync quick answer), and `GET /healthz` / `GET /readyz` — then drive the full lifecycle end-to-end with a **mock queue**, so the whole flow runs offline with no broker, no Redis, no port.

## 🧠 Why this matters

This is the one architectural pattern to internalize for any AI backend: **a thin, async, streaming API in front; durable workers behind.** The API's whole job is to *validate and dispatch in milliseconds* — accept a request, enqueue the slow stateful agent work, and hand back a `run_id` the client can poll or subscribe to. The agent run itself never happens inside the request handler, because anything that takes seconds there stalls every other request sharing the event loop (the trap from 25-01).

So the surface has two response styles, and choosing between them is the design. `POST /runs` is **async dispatch**: it returns *immediately* with `queued` and a `run_id` — not the answer. `POST /ask` is **sync**: for a short request, you wait and return the answer in one round trip. Everything heavy goes behind the queue boundary; the queue is what keeps the API thin.

## Objectives & prerequisites

**By the end you can:**

- Stand up the capstone routes: `POST /runs` (`201` + `run_id`), `GET /runs/{id}`, `GET /runs/{id}/stream` (SSE), `POST /ask`, and the `GET /healthz` / `GET /readyz` probes.
- Replace the real `run_agent.delay(...)` with a **mock task queue** (an in-memory job store + a background task) so the flow runs deterministically offline.
- Drive the lifecycle end-to-end: start a run → poll status → subscribe to the SSE stream → see it finish, asserting `201`/`200` and event order.
- Make `POST /runs` **idempotent** with an idempotency key so a retried "start" doesn't launch two runs.

**Prereqs:** `25-01-fastapi-from-zero.ipynb`, `25-02-streaming-sse-and-websockets.ipynb`. The Celery worker is *mocked* here; the real one is Ch 31. Packages: `fastapi`, `httpx`, `pydantic` — all in `requirements.txt`.

**Run first:** the Setup cell. Defaults to `MOCK=1`; the agent and queue are stubbed, progress is replayed from `data/run-progress.json`, so the notebook is **free, offline, deterministic** — no external broker, no bound port.

In [ ]:
# --- Setup -------------------------------------------------------------------
import asyncio
import json
import os
import random
import uuid
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # secrets (if any) come from the environment only

# MOCK=1 (default): agent + queue are stubbed -> free, offline, deterministic.
# MOCK=0 would run one short live agent step (see run_agent_steps). No broker either way.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(25)  # determinism; run_ids are made deterministic below in MOCK

MODEL = "claude-opus-4-8"  # the book's default model (only used if MOCK=0)
DATA_DIR = Path("data")    # tiny committed fixtures live here

# The scripted progress the mock worker emits while a run executes.
PROGRESS = json.loads((DATA_DIR / "run-progress.json").read_text(encoding="utf-8"))["progress"]

if MOCK:
    print(f"MOCK mode: mock queue + scripted worker ({len(PROGRESS)} progress events). Offline, free, no broker.")
else:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
            "Set it in your environment/.env, or use COMPANION_MOCK=1."
        )
    print(f"LIVE mode: the worker will make a short call to {MODEL}. Still no external broker.")

## 1 · The mock queue: an in-memory run store + a background task

In production, `POST /runs` calls `run_agent.delay(run_id, question)` and a **Celery** worker picks it up (Ch 31). Here we stand in for the broker with a dict and an `asyncio` task: `RunStore` holds each run's status, progress log, and result; `enqueue_run` schedules the mock worker to fill it in *after* the handler has already returned. Same boundary, same contract — no Redis, no separate process.

The key property we preserve: **`enqueue` returns instantly; the work happens later, off the request path.**

In [ ]:
class RunStore:
    """In-memory stand-in for the runs table + result store (real one: Ch 28/31)."""

    def __init__(self):
        self.runs: dict[str, dict] = {}
        self.idempotency: dict[str, str] = {}  # idempotency_key -> run_id
        self._counter = 0

    def create(self, question: str, max_steps: int) -> str:
        # Deterministic ids in MOCK so output is stable; uuid4 in live mode.
        if MOCK:
            self._counter += 1
            run_id = f"run-{self._counter:04d}"
        else:
            run_id = f"run-{uuid.uuid4().hex[:8]}"
        self.runs[run_id] = {
            "run_id": run_id,
            "status": "queued",
            "question": question,
            "max_steps": max_steps,
            "progress": [],
            "result": None,
        }
        return run_id

    def get(self, run_id: str) -> dict | None:
        return self.runs.get(run_id)


async def run_agent_steps(run: dict):
    """Yield progress events for a run. MOCK replays the fixture; live would call
    the agent and yield real step events. This is what the worker executes."""
    if MOCK:
        for ev in PROGRESS:
            await asyncio.sleep(0)  # real yield point; instant + deterministic
            yield ev
        return
    import anthropic  # imported only on the live path

    client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY
    yield {"type": "status", "phase": "started", "message": "calling the model"}
    resp = await client.messages.create(
        model=MODEL, max_tokens=256,
        messages=[{"role": "user", "content": run["question"]}],
    )
    yield {"type": "done", "message": "complete", "answer": resp.content[0].text}


async def _worker(store: RunStore, run_id: str):
    """The mock Celery worker: drains run_agent_steps into the store."""
    run = store.get(run_id)
    run["status"] = "running"
    async for ev in run_agent_steps(run):
        run["progress"].append(ev)
        if ev["type"] == "done":
            run["result"] = ev["answer"]
            run["status"] = "completed"


def enqueue_run(store: RunStore, run_id: str):
    """Stand-in for run_agent.delay(...): schedule the worker, return immediately."""
    asyncio.create_task(_worker(store, run_id))  # off the request path


print("mock queue ready (no broker, no Redis):", RunStore.__name__)

## 2 · 🔧 The routes: thin in front, queue behind

Now the API. Note how little each handler does. `POST /runs` creates a run, enqueues it, and returns `201` with `queued` — it does **not** wait for the answer. `POST /ask` is the sync exception for short work. `GET /runs/{id}` reads status. `GET /runs/{id}/stream` replays the run's progress as the SSE frames you built in 25-02. `healthz`/`readyz` are the Ch 28 probes: liveness vs readiness. The store lives on `app.state` (the lifespan pattern from 25-01), and DI hands it to each handler.

In [ ]:
from contextlib import asynccontextmanager

from fastapi import Depends, FastAPI, Header, HTTPException, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field


class AskRequest(BaseModel):
    question: str
    max_steps: int = Field(default=8, ge=1, le=32)


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.store = RunStore()      # one shared store per process
    app.state.ready = True            # readiness flag the probe reports
    yield
    app.state.ready = False           # graceful: stop accepting traffic


app = FastAPI(title="Capstone backend API (Ch 25 §25.5)", lifespan=lifespan)


def get_store(req: Request) -> RunStore:  # DI: hand the shared store to handlers
    return req.app.state.store


def sse_frame(payload: dict) -> str:
    return f"data: {json.dumps(payload)}\n\n"


@app.post("/runs", status_code=201)
async def start_run(
    req: AskRequest,
    store: RunStore = Depends(get_store),
    idempotency_key: str | None = Header(default=None),
):
    # Idempotency (Ch 24): a retried 'start' with the same key reuses the run.
    if idempotency_key and idempotency_key in store.idempotency:
        run_id = store.idempotency[idempotency_key]
        return {"run_id": run_id, "status": store.get(run_id)["status"], "reused": True}
    run_id = store.create(req.question, req.max_steps)
    if idempotency_key:
        store.idempotency[idempotency_key] = run_id
    enqueue_run(store, run_id)        # background (real: run_agent.delay, Ch 31)
    return {"run_id": run_id, "status": "queued"}


@app.get("/runs/{run_id}")
async def get_run(run_id: str, store: RunStore = Depends(get_store)):
    run = store.get(run_id)
    if run is None:
        raise HTTPException(status_code=404, detail=f"no such run: {run_id}")
    return {"run_id": run_id, "status": run["status"], "result": run["result"]}


@app.get("/runs/{run_id}/stream")
async def stream_run(run_id: str, store: RunStore = Depends(get_store)):
    run = store.get(run_id)
    if run is None:
        raise HTTPException(status_code=404, detail=f"no such run: {run_id}")

    async def events():
        sent = 0
        # Stream progress as it lands until the run is terminal.
        while True:
            progress = run["progress"]
            while sent < len(progress):
                yield sse_frame(progress[sent])
                sent += 1
            if run["status"] in {"completed", "failed"} and sent >= len(run["progress"]):
                break
            await asyncio.sleep(0)  # let the worker make progress
        yield "data: [DONE]\n\n"

    return StreamingResponse(events(), media_type="text/event-stream")


@app.post("/ask")
async def ask(req: AskRequest, store: RunStore = Depends(get_store)):
    # Sync path: for a short request, run it inline and return the answer now.
    run_id = store.create(req.question, req.max_steps)
    await _worker(store, run_id)       # awaited here -> we have the answer
    return {"answer": store.get(run_id)["result"]}


@app.get("/healthz")
async def healthz():
    return {"status": "ok"}            # liveness: the process is up


@app.get("/readyz")
async def readyz(req: Request):
    # readiness: are dependencies wired and are we accepting traffic? (Ch 28)
    if not getattr(req.app.state, "ready", False):
        raise HTTPException(status_code=503, detail="not ready")
    return {"status": "ready"}


print("routes:", sorted({r.path for r in app.routes if getattr(r, 'path', '').startswith(('/runs', '/ask', '/health', '/ready'))}))

## 3 · 🔮 Predict: `POST /runs` vs `POST /ask`

Both endpoints take the same `AskRequest`. But they embody the two dispatch styles, and the *immediate* response is the tell.

**Predict before running:** what comes back in the body of `POST /runs` the instant it returns — the agent's answer, or something else? And how does that differ from `POST /ask`? (Glance at the handlers above if you need to.) Decide, then run.

In [ ]:
from fastapi.testclient import TestClient

# TestClient as a context manager fires the lifespan (startup/shutdown).
with TestClient(app) as client:
    runs_resp = client.post("/runs", json={"question": "What is the refund window?"})
    ask_resp = client.post("/ask", json={"question": "What is the refund window?"})

    print("POST /runs ->", runs_resp.status_code, runs_resp.json())
    print("POST /ask  ->", ask_resp.status_code, ask_resp.json())

**What you just saw.** `POST /runs` returned **`201` with `{run_id, status: "queued"}`** — *not* the answer. It validated and dispatched in milliseconds; the agent work is still running behind the queue. `POST /ask` returned **`200` with the answer**, because it's the sync path for short requests and we awaited the work inline. That contrast — *async dispatch returns a handle; sync returns the result* — is the core of the whole surface. The slow, stateful work belongs behind the queue, where it can't stall the API.

## 4 · Drive the full lifecycle: start → poll → stream → finish

Now the real client journey. Start a run, **poll** `GET /runs/{id}` until the worker has had a chance to run, then **subscribe** to the SSE progress stream and watch the scripted events arrive in order, ending with the answer. We assert the status codes and the event order so this doubles as the notebook's CI check.

In [ ]:
import httpx


async def lifecycle():
    transport = httpx.ASGITransport(app=app)  # in-process; no port
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        # 1) Start the run.
        start = await ac.post("/runs", json={"question": "refund window?"})
        assert start.status_code == 201, start.status_code
        run_id = start.json()["run_id"]
        print(f"started {run_id}: {start.json()}")

        # 2) Poll status (let the background worker make progress).
        for _ in range(50):
            await asyncio.sleep(0)
            status = (await ac.get(f"/runs/{run_id}")).json()["status"]
            if status in {"running", "completed"}:
                break
        print(f"polled status: {status}")

        # 3) Subscribe to the SSE progress stream until [DONE].
        seen = []
        async with ac.stream("GET", f"/runs/{run_id}/stream") as resp:
            assert resp.status_code == 200, resp.status_code
            async for line in resp.aiter_lines():
                if not line.startswith("data: "):
                    continue
                data = line[len("data: "):]
                if data == "[DONE]":
                    break
                ev = json.loads(data)
                seen.append(ev["type"])
                print("  stream:", ev.get("message", ev))

        # 4) Final state.
        final = (await ac.get(f"/runs/{run_id}")).json()
        print("final:", final)
        return seen, final


seen, final = await lifecycle()
assert final["status"] == "completed", final
assert seen[-1] == "done", seen           # the stream ended on the 'done' event
assert final["result"], "expected an answer"
print("\nlifecycle OK — event order:", seen)

**What you just saw.** One run moved through `queued → running → completed`; the SSE stream replayed its progress events in order and ended on `done` carrying the answer — and every status code asserted. That's the capstone's contract working end-to-end, entirely offline. The frontend (Ch 38) is just a nicer renderer over this exact sequence of frames.

## 5 · ⚠️ Pitfall: doing the heavy work *inside* the request handler

The whole point of the queue is that the slow agent run happens **off** the request path. The tempting shortcut is to skip the queue and just `await` the agent inside `POST /runs` — it's fewer moving parts and it works in a demo. It also re-introduces the 25-01 trap at the worst possible spot: a multi-second run now holds the handler (and the event loop) the entire time, so concurrent requests queue up behind it.

Let's make it concrete. We register a `/runs-blocking` that does the agent work inline (a few hundred ms), then fire several concurrent requests at both it and the proper queued `/runs`, and time the totals.

In [ ]:
import time

WORK = 0.15  # seconds the 'agent' takes
N = 5


@app.post("/runs-blocking")
async def runs_blocking(req: AskRequest, store: RunStore = Depends(get_store)):
    # ANTI-PATTERN: run the slow work in the handler instead of enqueuing it.
    await asyncio.sleep(WORK)  # stands in for the multi-second agent run
    run_id = store.create(req.question, req.max_steps)
    return {"run_id": run_id, "status": "completed"}


async def hammer(path: str) -> float:
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        start = time.perf_counter()
        await asyncio.gather(*(ac.post(path, json={"question": "q"}) for _ in range(N)))
        return time.perf_counter() - start


queued_total = await hammer("/runs")            # enqueue + return -> near-instant
blocking_total = await hammer("/runs-blocking")  # work held in the handler

print(f"{N} concurrent starts:")
print(f"  POST /runs (queued)   : {queued_total:5.2f}s   (thin: validate + dispatch only)")
print(f"  POST /runs-blocking   : {blocking_total:5.2f}s   (heavy work in the handler)")
print("\nThe queue is what keeps the API thin and its latency flat under load.")

**What you just saw.** The queued endpoint returned almost instantly regardless of how many requests arrived together — it only validates and dispatches. The blocking variant held the work in the handler, so the requests piled up and the total climbed. In production the agent run is *seconds*, not 150ms, which turns this from a slowdown into an outage. **The queue boundary isn't an optimization; it's what makes a thin API possible.**

## 🎯 Senior lens: idempotent `POST /runs`

`POST /runs` *starts* something — and on the open internet, starts get retried. The client's network library retries on a dropped connection, a proxy replays a request, a user double-clicks. Without protection, each retry launches *another* agent run: duplicate work, duplicate spend, and a fan-out of confusing duplicate results. The fix is the **idempotency key** from Ch 24: the client sends a stable `Idempotency-Key` header, and the server records `key → run_id` on first use. A retry with the same key returns the *original* `run_id` instead of creating a new run.

This is where retry-safety meets the real endpoint. Reads (`GET /runs/{id}`) are naturally idempotent; the *create* is the dangerous one, and an idempotency key is what makes "start the run" safe to call twice. We wired it into the handler above — let's prove it.

In [ ]:
with TestClient(app) as client:
    headers = {"Idempotency-Key": "client-abc-123"}
    first = client.post("/runs", json={"question": "refund?"}, headers=headers)
    retry = client.post("/runs", json={"question": "refund?"}, headers=headers)  # same key

    print("first start:", first.json())
    print("retry      :", retry.json())
    assert first.json()["run_id"] == retry.json()["run_id"], "retry must reuse the run"
    print("\nSame run_id -> the retried 'start' did NOT launch a second run.")

## 📋 Production-readiness checklist

Before this surface is real (the next pillars make each line true):

- [ ] **Real queue** — swap the in-memory store + background task for Celery + a broker (Ch 31); the worker is a separate process that survives an API restart.
- [ ] **Persistent run store** — Postgres, not a dict, so status/results outlive the process (Ch 28).
- [ ] **Auth on every route** — `Depends` an auth dependency; `POST /runs` and `/ask` cost money (Ch 26).
- [ ] **`readyz` checks real dependencies** — broker + DB reachable, not just a flag (Ch 28).
- [ ] **Idempotency persisted + TTL'd** — keys in the store, not in memory, with expiry (Ch 24).
- [ ] **Per-stream error frames** — the `try/except/finally` hygiene from 25-02 on `GET /runs/{id}/stream`.
- [ ] **Timeouts + cancellation** — a run can be cancelled and won't stream forever.

## Recap

- **The surface is small and thin:** `POST /runs` (`201` + `run_id`, *queued not answered*), `GET /runs/{id}` (status/result), `GET /runs/{id}/stream` (SSE progress), `POST /ask` (sync quick answer), `GET /healthz` + `GET /readyz`.
- **Async dispatch vs sync:** `/runs` returns a *handle* and runs the work behind the queue; `/ask` returns the *result* inline for short requests.
- **The queue boundary keeps the API thin** — heavy agent work inside the handler stalls concurrent requests; off the request path it doesn't.
- **`POST /runs` must be idempotent** — an `Idempotency-Key` maps a retried start to the original `run_id` so retries don't launch duplicate runs.
- **It all ran offline** — mock queue (no broker/Redis), in-process ASGI (no port), scripted progress — and still matches the book's route names, status codes, and SSE frame shape.

## Exercises

Each one *changes something* and asks you to predict first. (Solutions land in `solutions/` in Phase 2.)

1. **404 on a missing run.** Call `GET /runs/run-does-not-exist`. Predict the status code and body, then confirm — and explain why a missing *resource* is a `404` while a malformed *body* (25-01) was a `422`.
2. **Cancel a run.** Add `POST /runs/{id}/cancel` that flips a run's status to `cancelled` and makes the worker stop emitting progress. Predict what the SSE stream sends after a cancel mid-run, then implement and verify it ends cleanly on `[DONE]`.
3. **Failure path.** Make the mock worker raise on a question containing the word `"boom"`, set `status="failed"`, and emit an `error` progress event. Predict the final `GET /runs/{id}` body and the last frame of the SSE stream, then assert both.

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here


## Next

You built the toy. Here's the real one.

- 🔧 **Template (the production scaffold of this surface):** [`../../../templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/) — the typed routes, lifespan, SSE, and health probes these notebooks built, assembled as a real service. You built the toy; that's the real one.
- 🎓 **Capstone:** this becomes [`capstone/app/`](../../../capstone/app/); checkpoint `checkpoints/ch25-backend-api`. The durable queue handoff (mocked here) lands fully in Ch 31 (`capstone/workers/`).
- 🧱 **Blueprint:** the agent behind `run_agent`/`stream_agent` is [`../../../blueprints/agent-loop/`](../../../blueprints/agent-loop/).
- 📖 **Book:** §25.5 (the Build); DI + health probes in §28, Celery workers in §31, the streaming frontend in §38.